In [1]:
# Importar librerías
import pandas as pd
from src.config import data_folder
from src.extract import *

In [2]:
# Obtener lista de tickers del S&P 500 desde el fichero constituents.csv
# Si no existe el fichero, lo descarga desde GitHub
# Para actualizar la lista de componentes, cambiar force_update a True, ejecutar y volver a dejar en False
ruta_archivo = descargar_constituents(force_update=False) 

# Cargar datos
df_tickers = pd.read_csv(ruta_archivo)
df_tickers.info()

Usando archivo constituents.csv local.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Symbol                 503 non-null    object
 1   Security               503 non-null    object
 2   GICS Sector            503 non-null    object
 3   GICS Sub-Industry      503 non-null    object
 4   Headquarters Location  503 non-null    object
 5   Date added             503 non-null    object
 6   CIK                    503 non-null    int64 
 7   Founded                503 non-null    object
dtypes: int64(1), object(7)
memory usage: 31.6+ KB


In [3]:
# Limpieza previa de df_tickers
df_tickers_clean = limpieza_tickers(df_tickers)

# Lista de tickers
tickers_list = df_tickers_clean["Ticker"].tolist()
df_tickers_clean.head()

,Ticker,Sector,DateAdded
0,MMM,Industrials,1957-03-04
1,AOS,Industrials,2017-07-26
2,ABT,HealthCare,1957-03-04
3,ABBV,HealthCare,2012-12-31
4,ACN,InformationTechnology,2011-07-06


In [4]:
# Extraer datos de SimFin (trimestrales de los últimos 4 años con un lag de 1 año)
df_simfin_raw = extraer_simfin(tickers_list)
df_simfin_raw.shape

Dataset "us-income-quarterly" on disk (1 days old).
- Loading from disk ... Done!
Dataset "us-balance-quarterly" on disk (1 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-quarterly" on disk (1 days old).
- Loading from disk ... Done!
Dataset "us-income-banks-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-balance-banks-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-banks-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-income-insurance-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-balance-insurance-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-insurance-quarterly" on disk (0 days old).
- Loading from disk ... Done!


(6692, 82)

In [5]:
# Identificar los tickers que SimFin devolvió exitosamente
tickers_simfin = df_simfin_raw['Ticker'].unique().tolist()
print(f"Tickers obtenidos de SimFin: {len(tickers_simfin)}")
print(f"Tickers descartados por limitación: {len(tickers_list) - len(tickers_simfin)}")

Tickers obtenidos de SimFin: 384
Tickers descartados por limitación: 119


In [6]:
# Se acota el Universo de tickers, limitado por aquellos que tienen datos disponibles en la versión gratuita de SimFin
# Se guarda el fichero del universo de tickers restringido
tickers_universe = pd.DataFrame(tickers_simfin, columns=["Ticker"])
tickers_universe.to_csv(f'{data_folder}/tickers_universe.csv')

In [7]:
# Extraer precios de los tickers y del índice SPY (demora unos 2 minutos)
df_prices = extraer_precios(tickers_simfin)

# Se extrae precio del Índice de Mercado para usar en cálculos y se guarda en fichero
df_index = extraer_precios(['SPY'])
df_index.to_parquet(f"{data_folder}/market_index.parquet")

df_prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9548 entries, 0 to 9547
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       9548 non-null   datetime64[ns]
 1   Open       9548 non-null   float64       
 2   Close      9548 non-null   float64       
 3   Dividends  9548 non-null   float64       
 4   Ticker     9548 non-null   object        
dtypes: datetime64[ns](1), float64(3), object(1)
memory usage: 373.1+ KB


In [8]:
# Unir df_prices y df_tickers
df_merged = pd.merge(
    df_prices,
    df_tickers_clean,
    on= 'Ticker',
    how= 'left'
)
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9548 entries, 0 to 9547
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       9548 non-null   datetime64[ns]
 1   Open       9548 non-null   float64       
 2   Close      9548 non-null   float64       
 3   Dividends  9548 non-null   float64       
 4   Ticker     9548 non-null   object        
 5   Sector     9548 non-null   object        
 6   DateAdded  9548 non-null   object        
dtypes: datetime64[ns](1), float64(3), object(3)
memory usage: 522.3+ KB


In [9]:
# Extraer datos financieros de yfinance (últimos 4 reportes trimestrales)
"""
yfinance no incluye la fecha de publicación, sólo la fecha de los Estados Financieros

Para evitar el "LookAhead" bias, se ofrecen 2 alternativas con el parametro "aproximar_fechas":

aproximar_fechas = True: 
-- Se estima la fecha de publicación con una regla de 30 días por defecto (editable en src/config.py)
-- Demora de ejecución: Aproximadamente 8 minutos

aproximar_fechas = False: 
-- Obtiene la fecha real de publicación usando yf.earnings_dates
-- Se aplica la regla de 30 días sólo si no se encuentra
-- Demora de ejecución: puede demorar 20 minutos o más
"""

df_yfinance = extraer_financials(tickers_simfin, aproximar_fechas = True)
df_yfinance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1536 entries, 0 to 1535
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Date                       1536 non-null   datetime64[ns]
 1   Total Revenue              1532 non-null   float64       
 2   Gross Profit               1425 non-null   float64       
 3   Operating Income           1436 non-null   float64       
 4   Net Income                 1532 non-null   float64       
 5   EBITDA                     1433 non-null   float64       
 6   Basic Average Shares       1506 non-null   float64       
 7   Cash And Cash Equivalents  1531 non-null   float64       
 8   Current Debt               1145 non-null   float64       
 9   Long Term Debt             1418 non-null   float64       
 10  Total Debt                 1507 non-null   float64       
 11  Stockholders Equity        1528 non-null   float64       
 12  Total 

In [10]:
# Definir columnas a mantener en simFin para que coincidan y estandarizar antes de unir
cols_yfinance = df_yfinance.columns
df_simfin_clean = estandarizar_simfin(df_simfin_raw, cols_yfinance)

df_financials_completo = unir_financials(df_yfinance, df_simfin_clean)
df_financials_completo.info()

Se han encontrado 0 filas con Ticker y Date solapados.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7999 entries, 0 to 7998
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Ticker                     7999 non-null   object        
 1   Date                       7990 non-null   datetime64[ns]
 2   Total Revenue              7988 non-null   float64       
 3   Gross Profit               7179 non-null   float64       
 4   Operating Income           7892 non-null   float64       
 5   Net Income                 7988 non-null   float64       
 6   EBITDA                     4404 non-null   float64       
 7   Basic Average Shares       7956 non-null   float64       
 8   Cash And Cash Equivalents  7966 non-null   float64       
 9   Current Debt               5959 non-null   float64       
 10  Long Term Debt             7439 non-null   float64       
 11  Total Debt    

In [11]:
# Unir dataframe de precios con datos financieros
df_final = pd.merge(
    df_merged, 
    df_financials_completo, 
    on=['Date', 'Ticker'],
    how='left'
)
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9548 entries, 0 to 9547
Data columns (total 26 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Date                       9548 non-null   datetime64[ns]
 1   Open                       9548 non-null   float64       
 2   Close                      9548 non-null   float64       
 3   Dividends                  9548 non-null   float64       
 4   Ticker                     9548 non-null   object        
 5   Sector                     9548 non-null   object        
 6   DateAdded                  9548 non-null   object        
 7   Total Revenue              7883 non-null   float64       
 8   Gross Profit               7079 non-null   float64       
 9   Operating Income           7787 non-null   float64       
 10  Net Income                 7883 non-null   float64       
 11  EBITDA                     4358 non-null   float64       
 12  Basic 

In [12]:
# Limpieza final, con forward fill para datos financieros (se llenan datos mensuales con el último dato trimestral)
df_final_clean = limpieza_final(df_final)
df_final_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4489 entries, 0 to 4488
Data columns (total 26 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Date                    4489 non-null   datetime64[ns]
 1   Open                    4489 non-null   float64       
 2   Close                   4489 non-null   float64       
 3   Dividends               4489 non-null   float64       
 4   Ticker                  4489 non-null   object        
 5   Sector                  4489 non-null   object        
 6   DateAdded               4489 non-null   object        
 7   TotalRevenue            4489 non-null   float64       
 8   GrossProfit             4243 non-null   float64       
 9   OperatingIncome         4489 non-null   float64       
 10  NetIncome               4489 non-null   float64       
 11  EBITDA                  4489 non-null   float64       
 12  BasicAverageShares      4485 non-null   float64 

In [13]:
# Guardar datos extraidos en fichero raw_data
df_final_clean.to_parquet(f"{data_folder}/raw_data.parquet")
print("Fichero guardado en la carpeta",data_folder)

Fichero guardado en la carpeta data
